# Minería de Datos — Turismo y Ocupación Hotelera en Risaralda
**Proyecto:** José Luis Brito Henao · Luis Eduardo Trujillo Agudelo  
**Contexto:** Terremoto de Pereira (Risaralda) — agosto de 2026  
**Dataset:** `dataset_cruzado.xlsx` (EMA-DANE + RNT)  
**Pregunta de negocio:** ¿Cómo cambió la ocupación hotelera en el Eje Cafetero (Risaralda) tras el terremoto de agosto de 2026, comparado con otras regiones y con la tendencia pre-pandemia?

**Estructura:**
1. Calidad de datos (Unidad 3)
2. Análisis exploratorio — EDA (Unidad 3 parte 2)
3. Ingeniería de características y pipeline (Unidad 4)
4. Modelo predictivo y evaluación (Unidad 6)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             classification_report)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_excel("dataset_cruzado.xlsx")
print("Shape:", df.shape)
print(df.dtypes)
df.head()

## 1. Calidad de datos
Objetivo: corregir tipos, detectar nulos/duplicados y tratar al menos 2 inconsistencias reales, documentando la decisión y el porqué.

In [ ]:
print("=== Tipos de dato ===")
print(df.dtypes)
print("\n=== Nulos por columna ===")
print(df.isna().sum())
print("\n=== Duplicados exactos ===")
print("Duplicados:", df.duplicated().sum())
print("\n=== Cardinalidad categóricas ===")
for c in df.select_dtypes(include="object").columns:
    print(f"{c}: {df[c].nunique()} únicos -> {df[c].unique()[:8]}")

In [ ]:
df_clean = df.copy()

# Ocupación: texto con % -> float 0-1
df_clean["ocupacion"] = (
    df_clean["ocupacion"].astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df_clean["ocupacion"] = pd.to_numeric(df_clean["ocupacion"], errors="coerce") / 100

# Unificar mes y construir fecha
mes_map = {"Enero":1,"Febrero":2,"Marzo":3,"Abril":4,"Mayo":5,"Junio":6,
           "Julio":7,"Agosto":8,"Septiembre":9,"Octubre":10,"Noviembre":11,"Diciembre":12}
col_mes = "Meses" if "Meses" in df_clean.columns else "Mes"
df_clean = df_clean.rename(columns={col_mes: "mes_nombre"})
df_clean["mes"] = df_clean["mes_nombre"].map(mes_map)

col_anio = "Año" if "Año" in df_clean.columns else "anio"
df_clean = df_clean.rename(columns={col_anio: "anio"})
df_clean["anio"] = pd.to_numeric(df_clean["anio"], errors="coerce")

df_clean["fecha"] = pd.to_datetime(
    dict(year=df_clean["anio"], month=df_clean["mes"], day=1), errors="coerce"
)

for c in ["establecimientos", "habitaciones"]:
    if c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce").fillna(0).astype(int)

print("Tipos corregidos:")
print(df_clean.dtypes)
print("\nNulos en fecha:", df_clean["fecha"].isna().sum())
print("Nulos en ocupacion:", df_clean["ocupacion"].isna().sum())

In [ ]:
# Atípico 1: ocupación fuera de [0,1]
fuera_rango = df_clean[(df_clean["ocupacion"] < 0) | (df_clean["ocupacion"] > 1)]
print(f"Ocupaciones fuera de [0,1]: {len(fuera_rango)}")
df_clean.loc[fuera_rango.index, "ocupacion"] = np.nan

# Bandera pandemia (valor extremo real, se mantiene)
df_clean["es_pandemia"] = ((df_clean["anio"] == 2020) | (df_clean["anio"] == 2021)).astype(int)

# Normalizar regiones
print("\nRegiones ANTES:", sorted(df_clean["region"].dropna().unique()))
df_clean["region"] = df_clean["region"].astype(str).str.strip().str.title()
print("Regiones DESPUÉS:", sorted(df_clean["region"].unique()))

print("\nShape final:", df_clean.shape)

### Decisiones de calidad de datos

| Problema | Decisión | Justificación |
|---|---|---|
| Ocupación como texto con `%` | Convertida a float 0–1 | Rompe cálculos y gráficos |
| `Meses` vs `Mes` | Unificada a `mes` (1–12) | Consistencia entre archivos |
| Sin columna fecha | Construida `fecha` datetime | Permite derivar trimestre/franja |
| Ocupación fuera de [0,1] | Marcada como NaN | Imposible por definición |
| Abril 2020 (pandemia) | **Mantenido** y marcado | Valor extremo real, clave para la pregunta |
| Regiones con escritura inconsistente | Normalizadas con `.str.title()` | Evita duplicados |

## 2. Análisis exploratorio de datos (EDA)
Objetivo: describir la variable numérica principal y la categórica principal, construir ≥3 gráficos y redactar 4–5 hallazgos.

In [ ]:
print("=== Ocupación (describe) ===")
print(df_clean["ocupacion"].describe())
print("\nMedia:", df_clean["ocupacion"].mean())
print("Mediana:", df_clean["ocupacion"].median())

print("\n=== Frecuencia por región ===")
print(df_clean["region"].value_counts())

print("\n=== Ocupación promedio por región ===")
print(df_clean.groupby("region")["ocupacion"].mean().sort_values(ascending=False))

In [ ]:
df_clean["ocupacion"].dropna().hist(bins=20, color="#1f4e79", edgecolor="white")
plt.title("Distribución de la ocupación hotelera (%)")
plt.xlabel("Ocupación")
plt.ylabel("Frecuencia")
plt.show()

df_clean.boxplot(column="ocupacion", by="region", rot=90, figsize=(12,5))
plt.title("Ocupación por región")
plt.suptitle("")
plt.ylabel("Ocupación")
plt.show()

df_clean["region"].value_counts().plot(kind="bar", color="#1f4e79")
plt.title("Cantidad de registros por región")
plt.ylabel("Registros")
plt.xticks(rotation=90)
plt.show()

pivot = df_clean.pivot_table(index="fecha", columns="region", values="ocupacion", aggfunc="mean")
cols = [c for c in ["Eje Cafetero", "Bogotá D.C.", "Cartagena"] if c in pivot.columns]
if cols:
    pivot[cols].plot(figsize=(12,5))
    plt.title("Evolución mensual de ocupación — regiones clave")
    plt.ylabel("Ocupación")
    plt.axvline(pd.Timestamp("2020-03-01"), color="red", ls="--", label="Pandemia")
    plt.axvline(pd.Timestamp("2026-08-01"), color="orange", ls="--", label="Terremoto")
    plt.legend()
    plt.show()

### Historia de hallazgos

1. **La pandemia desplomó la ocupación:** abril 2020 cayó a 0% en San Andrés y <12% en todas las regiones.
2. **Cartagena y San Andrés** tienen la mayor ocupación estructural (58.7% y 53.6%). El Eje Cafetero queda en 42.3%, por debajo del promedio nacional (45.9%).
3. **El Eje Cafetero se recupera más lento:** julio 2022 llegó a 52.5%, pero julio 2026 ya bajó a 44.5%.
4. **Alta heterogeneidad regional:** de 31.1% (Golfo Morrosquillo) a 58.7% (Cartagena).
5. **El terremoto ocurre en un momento de ocupación ya descendente** en el Eje Cafetero; hay que medir contra la tendencia 2022–2026.

## 3. Ingeniería de características y pipeline reproducible
Objetivo: crear ≥3 variables derivadas, codificar categóricas, escalar la numérica principal y encapsular todo en una función reproducible.

In [ ]:
df_fe = df_clean.copy()

# (1) Trimestre
df_fe["trimestre"] = df_fe["fecha"].dt.quarter

# (2) Franja histórica
df_fe["franja_anio"] = np.select(
    [df_fe["anio"] == 2019, df_fe["anio"].isin([2020, 2021])],
    ["pre-pandemia", "pandemia"],
    default="post-pandemia"
)

# (3) Bandera terremoto
df_fe["es_terremoto"] = (df_fe["fecha"] >= "2026-08-01").astype(int)

# (4) Categoría de ocupación
df_fe["ocupacion_categoria"] = pd.cut(
    df_fe["ocupacion"], bins=[-0.01, 0.30, 0.50, 1.01],
    labels=["bajo", "medio", "alto"]
)

# (5) Temporada alta
df_fe["temporada_alta"] = df_fe["mes"].isin([7, 8, 12]).astype(int)

df_fe[["fecha","trimestre","franja_anio","es_terremoto","ocupacion_categoria","temporada_alta"]].head(10)

In [ ]:
df_fe = pd.get_dummies(
    df_fe,
    columns=["region", "franja_anio", "ocupacion_categoria"],
    prefix=["region", "franja", "ocup_cat"],
    drop_first=False
)
print("Columnas después de one-hot:", df_fe.shape[1])
print([c for c in df_fe.columns if c.startswith(("region_","franja_","ocup_cat_"))])

In [ ]:
col_min = df_fe["ocupacion"].min()
col_max = df_fe["ocupacion"].max()
df_fe["ocupacion_escalada"] = (df_fe["ocupacion"] - col_min) / (col_max - col_min)
print(df_fe[["ocupacion","ocupacion_escalada"]].describe())

In [ ]:
def preparar_features(data):
    """Pipeline reproducible: limpieza + derivadas + one-hot + escalado."""
    d = data.copy()

    d["ocupacion"] = (
        d["ocupacion"].astype(str)
        .str.replace("%","",regex=False).str.replace(",",".",regex=False).str.strip()
    )
    d["ocupacion"] = pd.to_numeric(d["ocupacion"], errors="coerce") / 100

    col_mes = "Meses" if "Meses" in d.columns else "mes_nombre"
    if col_mes in d.columns:
        mes_map = {"Enero":1,"Febrero":2,"Marzo":3,"Abril":4,"Mayo":5,"Junio":6,
                   "Julio":7,"Agosto":8,"Septiembre":9,"Octubre":10,"Noviembre":11,"Diciembre":12}
        d["mes"] = d[col_mes].map(mes_map)

    col_anio = "Año" if "Año" in d.columns else "anio"
    d["anio"] = pd.to_numeric(d[col_anio], errors="coerce")
    d["fecha"] = pd.to_datetime(dict(year=d["anio"], month=d["mes"], day=1), errors="coerce")

    for c in ["establecimientos","habitaciones"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0).astype(int)

    d["trimestre"] = d["fecha"].dt.quarter
    d["franja_anio"] = np.select(
        [d["anio"] == 2019, d["anio"].isin([2020,2021])],
        ["pre-pandemia","pandemia"], default="post-pandemia")
    d["es_terremoto"] = (d["fecha"] >= "2026-08-01").astype(int)
    d["ocupacion_categoria"] = pd.cut(d["ocupacion"], bins=[-0.01,0.30,0.50,1.01],
                                       labels=["bajo","medio","alto"])
    d["temporada_alta"] = d["mes"].isin([7,8,12]).astype(int)

    d = pd.get_dummies(d, columns=["region","franja_anio","ocupacion_categoria"],
                       prefix=["region","franja","ocup_cat"])

    mn, mx = d["ocupacion"].min(), d["ocupacion"].max()
    d["ocupacion_escalada"] = (d["ocupacion"] - mn) / (mx - mn) if mx > mn else 0.0
    return d

df_v1 = preparar_features(df_clean)
df_v2 = preparar_features(df_clean)
print("¿Reproducible?", df_v1.equals(df_v2))
print("Shape final:", df_v1.shape)

### Variables derivadas (≥3)

| Variable | Qué aporta |
|---|---|
| `trimestre` | Estacionalidad dentro del año |
| `franja_anio` | Pre-pandemia / pandemia / post-pandemia |
| `es_terremoto` | Meses post-agosto 2026 |
| `ocupacion_categoria` | Bajo/medio/alto para clasificación |
| `temporada_alta` | Julio, agosto, diciembre |

## 4. Modelo predictivo y evaluación
Objetivo: clasificar `ocupacion_categoria` (bajo/medio/alto), evaluar con matriz de confusión, precisión, recall y F1, y revisar fuga de datos y sobreajuste.

In [ ]:
cols_excluir = ["ocupacion","ocupacion_escalada","ocupacion_categoria",
                "mes_nombre","fecha","region","franja_anio",
                "Año","anio","Mes","Meses"]

features = [c for c in df_v1.columns if c not in cols_excluir and df_v1[c].dtype != "object"]
X = df_v1[features].fillna(0)
y = df_v1["ocupacion_categoria"].astype(str)

print("Features usadas:", len(features))
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

pred_train = modelo.predict(X_train)
pred_test  = modelo.predict(X_test)

print("Accuracy train:", round(accuracy_score(y_train, pred_train), 3))
print("Accuracy test :", round(accuracy_score(y_test, pred_test), 3))

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=X.columns)
print(importancias.sort_values(ascending=False).head(10))

top = importancias.sort_values(ascending=False).iloc[0]
if top > 0.8:
    print(f"⚠ ALERTA: una feature concentra {top:.0%} de la importancia. Revisar fuga.")
else:
    print(f"✔ Sin fuga evidente (top feature = {top:.0%}).")

In [ ]:
cm = confusion_matrix(y_test, pred_test)
print("Matriz de confusión:\n", cm)
print("\nClassification report:")
print(classification_report(y_test, pred_test, zero_division=0))

print("Precisión (weighted):", round(precision_score(y_test, pred_test, average="weighted", zero_division=0), 3))
print("Recall   (weighted):", round(recall_score(y_test, pred_test, average="weighted", zero_division=0), 3))
print("F1       (weighted):", round(f1_score(y_test, pred_test, average="weighted", zero_division=0), 3))

In [ ]:
acc_train = accuracy_score(y_train, pred_train)
acc_test  = accuracy_score(y_test, pred_test)
print(f"Accuracy train: {acc_train:.3f}")
print(f"Accuracy test : {acc_test:.3f}")
print(f"Diferencia    : {acc_train - acc_test:.3f}")

if acc_train - acc_test > 0.15:
    print("⚠ Posible sobreajuste.")
elif acc_train < 0.5 and acc_test < 0.5:
    print("⚠ Modelo débil.")
else:
    print("✔ Diferencia razonable.")

In [ ]:
for depth in [2, 4, 8, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test,  m.predict(X_test))
    print(f"max_depth={str(depth):>4} | train={tr:.3f} | test={te:.3f} | gap={tr-te:.3f}")

### Conclusiones del modelo

- Objetivo: clasificar ocupación en bajo/medio/alto.
- Modelo: árbol de decisión `max_depth=4`.
- Se excluyó `ocupacion` y `ocupacion_escalada` de las features para evitar fuga.
- Se comparó accuracy train vs test para detectar sobreajuste.
- Limitación: solo hay julio 2026 como referencia post-terremoto.